# Image Classification: The Complete Pipeline

In Lesson 05, we traced data through an MLP that classified Titanic passengers. Each passenger had a handful of curated features: class, sex, age, fare. The network learned weight combinations that predicted survival.

What happens when our "features" aren't columns in a spreadsheet, but the raw pixels of an image? That's what we'll explore today. Same MLP architecture, radically different data. Along the way we'll build a complete image classification pipeline, from loading messy web-scraped photos to evaluating what the model learned.

We'll classify photos of 5 bird species: eagle, flamingo, owl, parrot, and penguin. The data is real, messy, and forces us to deal with problems that neatly packaged datasets like CIFAR-10 hide from you.

In [ ]:
# Colab setup - run this cell first if you're on Google Colab
try:
    import google.colab
    print("Colab environment ready! All required packages are pre-installed.")
except ImportError:
    pass  # Not on Colab, no action needed

In [ ]:
# === SETUP (collapse this cell) ===
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision
import torchvision.transforms as transforms
from pathlib import Path
from PIL import Image
from collections import Counter
from sklearn.metrics import confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')

In [ ]:
# === VISUALIZATION HELPERS (collapse this cell) ===

CLASS_NAMES = ['eagle', 'flamingo', 'owl', 'parrot', 'penguin']
CLASS_COLORS = ['#e74c3c', '#e91e8c', '#f39c12', '#2ecc71', '#3498db']

def show_image_grid(images, titles=None, rows=2, cols=5, figsize=(14, 6)):
    """Show a grid of PIL images or tensors."""
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(images):
            img = images[i]
            if isinstance(img, torch.Tensor):
                if img.dim() == 3:
                    img = img.permute(1, 2, 0).numpy()
                    if img.min() < 0:
                        img = (img - img.min()) / (img.max() - img.min())
                    img = np.clip(img, 0, 1)
            ax.imshow(img)
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def show_channel_heatmaps(image, title=''):
    """Show RGB channels as separate heatmaps."""
    if isinstance(image, Image.Image):
        image = np.array(image)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(image)
    axes[0].set_title('Original')
    
    channel_names = ['Red', 'Green', 'Blue']
    cmaps = ['Reds', 'Greens', 'Blues']
    for i in range(3):
        im = axes[i+1].imshow(image[:, :, i], cmap=cmaps[i], vmin=0, vmax=255)
        axes[i+1].set_title(f'{channel_names[i]} channel')
        plt.colorbar(im, ax=axes[i+1], shrink=0.8)
    
    for ax in axes:
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


def show_flatten_visualization(crop):
    """Show a small image crop as RGB grids + flattened vector."""
    h, w = crop.shape[:2]
    channel_names = ['Red', 'Green', 'Blue']
    channel_bg = ['#fee0d2', '#d4edda', '#cce5ff']
    channel_fg = ['#a63603', '#155724', '#004085']

    fig = plt.figure(figsize=(16, 7))

    ax0 = fig.add_axes([0.02, 0.55, 0.12, 0.35])
    ax0.imshow(crop, interpolation='nearest')
    ax0.set_title(f'{h}×{w} crop\n(enlarged)', fontsize=11)
    ax0.set_xticks([]); ax0.set_yticks([])

    for ch in range(3):
        ax = fig.add_axes([0.18 + ch * 0.22, 0.55, 0.20, 0.35])
        data = crop[:, :, ch]
        ax.set_xlim(0, w); ax.set_ylim(0, h)
        ax.set_aspect('equal'); ax.invert_yaxis()
        for r in range(h):
            for c in range(w):
                ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=channel_bg[ch],
                                           edgecolor='gray', linewidth=0.5))
                ax.text(c + 0.5, r + 0.5, str(data[r, c]), ha='center', va='center',
                        fontsize=10, fontweight='bold', color=channel_fg[ch])
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'{channel_names[ch]} channel', fontsize=11, color=channel_fg[ch])

    ax_flat = fig.add_axes([0.05, 0.05, 0.88, 0.35])
    flat_vals, flat_cols = [], []
    for ch in range(3):
        for r in range(h):
            for c in range(w):
                flat_vals.append(crop[r, c, ch])
                flat_cols.append(channel_bg[ch])

    n = len(flat_vals)
    for i in range(n):
        ax_flat.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=flat_cols[i],
                                         edgecolor='gray', linewidth=0.5))
        ax_flat.text(i + 0.5, 0.5, str(flat_vals[i]), ha='center', va='center',
                     fontsize=7, fontweight='bold')

    for ch, label in enumerate(['R values', 'G values', 'B values']):
        ax_flat.annotate(label, xy=(h * w * ch + h * w / 2, -0.1),
                         ha='center', fontsize=9, color=channel_fg[ch])

    ax_flat.set_xlim(-0.5, n + 0.5); ax_flat.set_ylim(-0.5, 1.5)
    ax_flat.set_xticks([]); ax_flat.set_yticks([])
    total = h * w * 3
    ax_flat.set_title(f'Flattened: {h} × {w} × 3 = {total} numbers in a row  →  this is what nn.Flatten() does',
                      fontsize=11)
    plt.suptitle('From 2D Image to 1D Vector', fontsize=14, fontweight='bold', y=0.99)
    plt.show()


def plot_sigmoid_vs_softmax(logits, class_names, class_colors):
    """Side-by-side bar chart: sigmoid vs softmax outputs from same logits."""
    sigmoid_probs = torch.sigmoid(logits)
    softmax_probs = torch.softmax(logits, dim=0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    bars1 = axes[0].bar(class_names, sigmoid_probs.numpy(), color=class_colors,
                        edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars1, sigmoid_probs):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
    axes[0].set_ylim(0, 1.15)
    axes[0].set_ylabel('Probability')
    axes[0].set_title(f'5 Independent Sigmoids\nSum = {sigmoid_probs.sum():.2f}  (not a valid distribution!)',
                      fontsize=12, color='#c0392b')
    axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

    bars2 = axes[1].bar(class_names, softmax_probs.numpy(), color=class_colors,
                        edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars2, softmax_probs):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
    axes[1].set_ylim(0, 1.15)
    axes[1].set_ylabel('Probability')
    axes[1].set_title(f'Softmax\nSum = {softmax_probs.sum():.2f}  (proper probability distribution)',
                      fontsize=12, color='#27ae60')
    axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

    plt.suptitle(f'Same logits {logits.tolist()} — different output functions',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def plot_softmax_bars(probs, class_names, class_colors):
    """Horizontal bar chart of softmax probabilities."""
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(class_names[::-1], probs[::-1], color=class_colors[::-1],
                   edgecolor='white', linewidth=1.5, height=0.6)
    for bar, prob in zip(bars, probs[::-1]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{prob:.1%}', va='center', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1.0)
    ax.set_xlabel('Probability')
    ax.set_title('Softmax Output: Probabilities Compete', fontsize=13, fontweight='bold')
    ax.axvline(x=0.2, color='gray', linestyle=':', alpha=0.3)
    ax.text(0.2, -0.7, '← uniform (20% each)', fontsize=9, color='gray', ha='center')
    plt.tight_layout()
    plt.show()


def plot_training_curves(history):
    """Plot loss and accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history['train_loss'], label='Train', color='#3498db')
    axes[0].plot(history['val_loss'], label='Validation', color='#e74c3c')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history['train_acc'], label='Train', color='#3498db')
    axes[1].plot(history['val_acc'], label='Validation', color='#e74c3c')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def denormalize(tensor, mean, std):
    """Reverse normalization for display."""
    t = tensor.clone()
    for c in range(3):
        t[c] = t[c] * std[c] + mean[c]
    return torch.clamp(t, 0, 1)


def softmax_spreadsheet(logits, class_names, true_class_idx):
    """Jeremy-style spreadsheet: output → exp → softmax → actuals for one prediction."""
    import pandas as pd
    logits_np = np.array(logits, dtype=np.float64)
    exp_vals = np.exp(logits_np)
    softmax_vals = exp_vals / exp_vals.sum()
    actuals = [1 if i == true_class_idx else 0 for i in range(len(class_names))]
    loss = -np.log(softmax_vals[true_class_idx])

    df = pd.DataFrame({
        'output': logits_np,
        'exp': exp_vals,
        'softmax': softmax_vals,
        'actuals': actuals,
        'index': range(len(class_names)),
    }, index=class_names)

    sum_row = pd.DataFrame({
        'output': [np.nan], 'exp': [exp_vals.sum()],
        'softmax': [softmax_vals.sum()], 'actuals': [np.nan], 'index': [np.nan],
    }, index=['sum'])
    df = pd.concat([df, sum_row])

    def style_fn(row):
        styles = [''] * len(row)
        if row.name == 'sum':
            styles = ['font-weight: bold; border-top: 2px solid #333'] * len(row)
        elif row.name in class_names and class_names.index(row.name) == true_class_idx:
            for i, col in enumerate(row.index):
                if col in ['output', 'exp', 'softmax', 'actuals']:
                    styles[i] = 'background-color: #d4edda; font-weight: bold'
        return styles

    return df.style\
        .apply(style_fn, axis=1)\
        .format({'output': '{:+.2f}', 'exp': '{:.2f}', 'softmax': '{:.2f}',
                 'actuals': lambda x: '' if pd.isna(x) else f'{int(x)}',
                 'index': lambda x: '' if pd.isna(x) else f'{int(x)}'})\
        .set_caption(f'Prediction for "{class_names[true_class_idx]}" image  →  Loss = -log({softmax_vals[true_class_idx]:.2f}) = {loss:.3f}')


def scenario_comparison_table(scenarios, class_names):
    """Multiple predictions: softmax probabilities + loss, with color-coded correct class."""
    import pandas as pd
    rows = []
    true_classes = {}
    for desc, logits_list, true_class in scenarios:
        logits_np = np.array(logits_list, dtype=np.float64)
        probs = np.exp(logits_np) / np.exp(logits_np).sum()
        loss = -np.log(probs[true_class])
        row = {'Scenario': desc, 'True label': class_names[true_class]}
        for i, name in enumerate(class_names):
            row[name] = probs[i]
        row['Loss'] = loss
        rows.append(row)
        true_classes[desc] = true_class

    df = pd.DataFrame(rows).set_index('Scenario')

    def style_fn(row):
        styles = [''] * len(row)
        tc_name = class_names[true_classes[row.name]]
        for i, col in enumerate(row.index):
            if col == tc_name:
                val = row[col]
                if val > 0.5:
                    styles[i] = 'background-color: #d4edda; font-weight: bold'
                elif val > 0.2:
                    styles[i] = 'background-color: #fff3cd; font-weight: bold'
                else:
                    styles[i] = 'background-color: #f8d7da; font-weight: bold'
            if col == 'Loss':
                val = row[col]
                if val > 2:
                    styles[i] = 'background-color: #f8d7da; font-weight: bold; color: #721c24'
                elif val > 0.5:
                    styles[i] = 'background-color: #fff3cd; font-weight: bold'
                else:
                    styles[i] = 'background-color: #d4edda; font-weight: bold; color: #155724'
        return styles

    return df.style\
        .format({n: '{:.3f}' for n in class_names})\
        .format({'Loss': '{:.3f}'})\
        .apply(style_fn, axis=1)\
        .set_caption('Softmax probabilities + Cross-Entropy Loss across scenarios')

## Part 1: Meet the Data

Our dataset is a collection of bird photos scraped from the web. Unlike CIFAR-10 or Fashion-MNIST, these images aren't neatly packaged: they're different sizes, some might be corrupted, and the files have random UUID names instead of descriptive labels.

This is what real-world image data looks like. The folder structure tells us the class:

```
bird_data/
├── eagle/
│   ├── 01c56617-0b79-4d71-90e1-aac0816027cd.jpg
│   └── ...
├── flamingo/
├── owl/
├── parrot/
└── penguin/
```

Each subfolder name IS the label. This pattern is so common that PyTorch has a built-in loader for it called `ImageFolder`.

> **Note:** The bird dataset lives at `../inspiration/bird_data` (relative to this notebook). You need to run the inspiration notebook first to generate/download the data, or place the `bird_data/` folder there manually.

In [ ]:
DATA_DIR = Path('../inspiration/bird_data')

# Count images per class
class_counts = {}
for class_dir in sorted(DATA_DIR.iterdir()):
    if class_dir.is_dir():
        n_images = len(list(class_dir.glob('*')))
        class_counts[class_dir.name] = n_images
        print(f'{class_dir.name:10s}: {n_images} images')

print(f'{"TOTAL":10s}: {sum(class_counts.values())} images')

In [ ]:
# Class distribution - is it balanced?
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(class_counts.keys(), class_counts.values(), color=CLASS_COLORS)
for bar, count in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(count), ha='center', fontweight='bold')
ax.set_ylabel('Number of images')
ax.set_title('Class Distribution')
plt.show()

Not perfectly balanced. Penguins and flamingos have roughly half the images of owls. This is typical for web-scraped data. Some search terms just yield more results than others.

For now we'll work with what we have. If the model struggles with penguins specifically, class imbalance could be a factor.

In [ ]:
# Show 2 samples from each class
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for col, class_name in enumerate(CLASS_NAMES):
    class_dir = DATA_DIR / class_name
    image_files = sorted(class_dir.glob('*'))[:2]
    for row, img_path in enumerate(image_files):
        try:
            img = Image.open(img_path).convert('RGB')
            axes[row, col].imshow(img)
            axes[row, col].set_title(f'{class_name}\n{img.size[0]}x{img.size[1]}', fontsize=10)
        except Exception:
            axes[row, col].text(0.5, 0.5, 'FAILED', ha='center', va='center', fontsize=14, color='red')
        axes[row, col].axis('off')

plt.suptitle('Sample Images (notice the different sizes!)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Notice the image dimensions printed below each photo. They're all different: 800x800, 1024x768, 640x480... A neural network needs a fixed input size. We can't feed a 800-pixel-wide eagle and a 640-pixel-wide parrot into the same model.

With tabular data this wasn't a problem, every row had the same columns. With images, we need an extra step: **resizing everything to the same dimensions**.

In [ ]:
# Survey actual image sizes across the dataset
widths, heights = [], []
for class_name in CLASS_NAMES:
    for img_path in (DATA_DIR / class_name).glob('*'):
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
        except Exception:
            pass

print(f'Images surveyed: {len(widths)}')
print(f'Width  - min: {min(widths)}, max: {max(widths)}, median: {int(np.median(widths))}')
print(f'Height - min: {min(heights)}, max: {max(heights)}, median: {int(np.median(heights))}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='#3498db', alpha=0.7, edgecolor='white')
axes[0].set_title('Image Widths')
axes[0].set_xlabel('Pixels')
axes[1].hist(heights, bins=30, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[1].set_title('Image Heights')
axes[1].set_xlabel('Pixels')
plt.tight_layout()
plt.show()

In [ ]:
# What does a computer see? RGB channels.
sample_path = next((DATA_DIR / 'parrot').glob('*'))
sample_img = Image.open(sample_path).convert('RGB')
sample_img = sample_img.resize((200, 200))  # resize for cleaner display

show_channel_heatmaps(sample_img, 'What the Computer Sees: Three Color Channels')

Every pixel in a color image is described by three numbers: Red, Green, Blue, each ranging from 0 (none) to 255 (full intensity). The parrot's bright green feathers show up as high values in the Green channel and low values in Red and Blue.

A single 200x200 color image = 200 x 200 x 3 = **120,000 numbers**. Compare that to a Titanic passenger's 4 features. That's a 30,000x increase in input size.

We'll work with 128x128 images in this lesson, which gives us 128 x 128 x 3 = **49,152 features** per image. Still enormous compared to tabular data.

### From Image to Numbers

In our tabular lessons, every row was already a vector of features: `[Pclass, Sex, Age, Fare]`. The model could eat it directly. An image is a 3D block of numbers: height × width × channels. An MLP only accepts 1D vectors. So we need to **flatten** it — unroll that 3D block into a single long row of numbers.

Let's see exactly what that looks like.

In [ ]:
# Take a tiny 4x4 crop from a real bird image
sample_path = next((DATA_DIR / 'parrot').glob('*'))
sample_img = Image.open(sample_path).convert('RGB')
sample_img = sample_img.resize((128, 128))

# Crop a 4x4 patch from the center (where there's likely color variation)
crop = np.array(sample_img)[62:66, 62:66]  # shape: (4, 4, 3)

show_flatten_visualization(crop)

In [ ]:
# Scale up to the real thing
sample = torch.randn(1, 3, 128, 128)  # one image as a tensor
flatten = nn.Flatten()

print(f'Image tensor shape:     {sample.shape}')
print(f'After nn.Flatten():     {flatten(sample).shape}')
print(f'Total features:         {flatten(sample).shape[1]:,}')
print()
print('Compare to tabular data:')
print(f'  Titanic passenger:    [Pclass, Sex, Age, Fare]           →      4 features')
print(f'  Adult Income row:     [age, education, hours, ...]       →     14 features')
print(f'  Bird image (128×128): [R₀, G₀, B₀, R₁, G₁, B₁, ...]   → 49,152 features')
print()
print('Same idea. Just a LOT more features.')
print('And unlike Pclass or Age, individual pixel values are mostly meaningless on their own.')

## Part 2: Data Cleaning

Web-scraped data is messy. Some files might be corrupted, truncated, or not even real images (a .jpg file that's actually a WebP or HTML error page). In tabular ML we checked for missing values and outliers. For images, we need to check that every file actually loads.

This step is boring but essential. One corrupted image in your DataLoader will crash your training loop at some random batch, and you'll spend hours debugging.

In [ ]:
# Try loading every image, track failures
valid_paths = []
failed_paths = []

for class_name in CLASS_NAMES:
    class_dir = DATA_DIR / class_name
    for img_path in class_dir.glob('*'):
        try:
            with Image.open(img_path) as img:
                img.convert('RGB')  # Force full decode
                img.load()          # Actually read pixel data
            valid_paths.append((img_path, class_name))
        except Exception as e:
            failed_paths.append((img_path, str(e)))

print(f'Valid images:  {len(valid_paths)}')
print(f'Failed images: {len(failed_paths)}')

if failed_paths:
    print('\nFailed files:')
    for path, error in failed_paths[:10]:
        print(f'  {path.parent.name}/{path.name}: {error[:60]}')

In a production setting you'd also want to check for:
- **Duplicate images** (same photo scraped twice)
- **Wrong content** (a photo of a building in the "eagle" folder)
- **Too small images** (tiny thumbnails that won't resize well)

For now, removing corrupted files is sufficient. We'll build our dataset from the valid paths only.

## Part 3: Resizing and Transforms

In Lesson 06, our data pipeline was:

```
pandas DataFrame → handle missing values → encode categoricals → scale numericals → tensors
```

For images, PyTorch provides a different pipeline built around `torchvision.transforms`:

```
load image → resize → augment → convert to tensor → normalize
```

Each step is a transform, and we chain them together with `transforms.Compose()`. Think of it like a preprocessing pipeline where each image passes through a sequence of operations.

In [ ]:
# Problem: what happens when we resize a non-square image to a square?
sample_path = next((DATA_DIR / 'eagle').glob('*'))
original = Image.open(sample_path).convert('RGB')

# Three strategies
squashed = original.resize((128, 128))  # Distorts aspect ratio
cropped = transforms.CenterCrop(min(original.size))(original)
cropped = cropped.resize((128, 128))
resized_crop = transforms.Compose([
    transforms.Resize(160),        # Resize shortest edge to 160
    transforms.CenterCrop(128),    # Crop center 128x128
])(original)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(original)
axes[0].set_title(f'Original\n{original.size[0]}x{original.size[1]}')
axes[1].imshow(squashed)
axes[1].set_title('Squash to 128x128\n(distorted!)')
axes[2].imshow(cropped)
axes[2].set_title('Center crop → resize\n(loses edges)')
axes[3].imshow(resized_crop)
axes[3].set_title('Resize + center crop\n(best balance)')

for ax in axes:
    ax.axis('off')
plt.suptitle('Resizing Strategies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

The third approach, `Resize` (to slightly larger) then `CenterCrop` (to target size), is the standard strategy. It preserves aspect ratio and only loses a small border. During training, we'll swap `CenterCrop` for `RandomResizedCrop` which picks a random crop each time, effectively a form of augmentation.

Now let's look at `transforms.Compose()`, which chains multiple transforms together:

In [ ]:
# A basic transform pipeline
basic_transform = transforms.Compose([
    transforms.Resize(160),          # Resize shortest edge to 160px
    transforms.CenterCrop(128),      # Crop center 128x128
    transforms.ToTensor(),           # PIL Image → Tensor + scale 0-255 → 0-1
])

# Apply to a sample image
sample_path = next((DATA_DIR / 'flamingo').glob('*'))
original = Image.open(sample_path).convert('RGB')
transformed = basic_transform(original)

print(f'Before: PIL Image, size {original.size}')
print(f'After:  Tensor, shape {transformed.shape}')
print(f'        dtype: {transformed.dtype}')
print(f'        value range: [{transformed.min():.3f}, {transformed.max():.3f}]')

Notice what `ToTensor()` does:

1. Converts a PIL Image to a PyTorch tensor
2. Rearranges dimensions from (H, W, C) to **(C, H, W)** — PyTorch convention puts channels first
3. Scales pixel values from 0-255 integers to **0.0-1.0** floats

That scaling is our first normalization step. It's simpler than tabular data where each feature needed its own StandardScaler. For images, dividing by 255 puts all pixels on the same scale. But we'll go further in Part 4 with per-channel normalization.

In [ ]:
# Before vs after the transform pipeline
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(original)
axes[0].set_title(f'Original PIL Image\n{original.size[0]}x{original.size[1]}, values 0-255')

# To display a tensor we need (C,H,W) → (H,W,C)
axes[1].imshow(transformed.permute(1, 2, 0).numpy())
axes[1].set_title(f'After transforms\n{transformed.shape[1]}x{transformed.shape[2]}, values 0-1')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Building a Custom Dataset

PyTorch's built-in `ImageFolder` expects all images to load cleanly. Since we know some of ours might fail (web-scraped data), we'll build a simple custom Dataset that skips bad files. This also lets us use only our verified `valid_paths` from the cleaning step.

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, image_paths, class_names, transform=None):
        """
        image_paths: list of (path, class_name) tuples from our cleaning step
        class_names: ordered list of class names (index = label)
        transform: torchvision transform to apply
        """
        self.image_paths = image_paths
        self.class_to_idx = {name: i for i, name in enumerate(class_names)}
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path, class_name = self.image_paths[idx]
        image = Image.open(path).convert('RGB')
        label = self.class_to_idx[class_name]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label


# Quick test
test_dataset = BirdDataset(valid_paths, CLASS_NAMES, transform=basic_transform)
img, label = test_dataset[0]
print(f'Image shape: {img.shape}')
print(f'Label: {label} ({CLASS_NAMES[label]})')
print(f'Dataset size: {len(test_dataset)}')

This pattern, a class with `__len__` and `__getitem__`, is how every PyTorch Dataset works. The DataLoader will call `__getitem__` repeatedly with different indices to build batches. The transform gets applied each time an image is loaded, which means augmentations (Part 5) produce different results each epoch.

Compare this to tabular data where we loaded everything into a numpy array upfront. With images, we load and transform on-the-fly because keeping all images in memory (especially at full resolution) would be wasteful.

## Part 4: Per-Channel Normalization

In Lesson 06, we used `StandardScaler` to normalize each tabular feature to mean=0 and std=1. For images, we do the same thing but per color channel. Instead of normalizing "Age" and "Fare" separately, we normalize "Red pixels", "Green pixels", and "Blue pixels" separately.

Why? Each channel has a different distribution. Blue skies push the Blue channel higher. Green forests push Green higher. If we just divide by 255, the channels are on the same scale (0-1) but might have very different means and spreads. Per-channel normalization centers each channel around 0 with unit variance, which helps gradient-based optimization converge faster.

We need to compute the mean and standard deviation of each channel across our entire training set.

In [ ]:
# Compute per-channel mean and std across the entire dataset
# We use the basic_transform (resize + crop + ToTensor) without normalization
temp_dataset = BirdDataset(valid_paths, CLASS_NAMES, transform=basic_transform)

# Accumulate channel sums
channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
n_pixels = 0

for img, _ in temp_dataset:
    # img shape: (3, 128, 128)
    channel_sum += img.sum(dim=[1, 2])       # sum all pixels per channel
    channel_sq_sum += (img ** 2).sum(dim=[1, 2])
    n_pixels += img.shape[1] * img.shape[2]  # 128 * 128

channel_mean = channel_sum / n_pixels
channel_std = ((channel_sq_sum / n_pixels) - channel_mean ** 2).sqrt()

print('Per-channel statistics (after ToTensor 0-1 scaling):')
print(f'  Red:   mean={channel_mean[0]:.4f}, std={channel_std[0]:.4f}')
print(f'  Green: mean={channel_mean[1]:.4f}, std={channel_std[1]:.4f}')
print(f'  Blue:  mean={channel_mean[2]:.4f}, std={channel_std[2]:.4f}')

# Save for later use
BIRD_MEAN = channel_mean.tolist()
BIRD_STD = channel_std.tolist()
print(f'\nBIRD_MEAN = {[round(x, 4) for x in BIRD_MEAN]}')
print(f'BIRD_STD  = {[round(x, 4) for x in BIRD_STD]}')

In [ ]:
# Visualize the effect of normalization on pixel value distributions
sample_img = temp_dataset[0][0]  # (3, 128, 128), values 0-1

# Apply normalization manually
normalized = sample_img.clone()
for c in range(3):
    normalized[c] = (normalized[c] - BIRD_MEAN[c]) / BIRD_STD[c]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
channel_names = ['Red', 'Green', 'Blue']
colors = ['#e74c3c', '#2ecc71', '#3498db']

for c in range(3):
    # Before normalization (0-1 range)
    axes[0, c].hist(sample_img[c].flatten().numpy(), bins=50, color=colors[c], alpha=0.7, edgecolor='white')
    axes[0, c].set_title(f'{channel_names[c]} - Before')
    axes[0, c].set_xlim(-0.1, 1.1)
    axes[0, c].axvline(BIRD_MEAN[c], color='black', linestyle='--', label=f'mean={BIRD_MEAN[c]:.2f}')
    axes[0, c].legend(fontsize=9)
    
    # After normalization (centered around 0)
    axes[1, c].hist(normalized[c].flatten().numpy(), bins=50, color=colors[c], alpha=0.7, edgecolor='white')
    axes[1, c].set_title(f'{channel_names[c]} - After')
    axes[1, c].axvline(0, color='black', linestyle='--', label='mean≈0')
    axes[1, c].legend(fontsize=9)

axes[0, 0].set_ylabel('Before\n(0 to 1)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('After\n(centered)', fontsize=12, fontweight='bold')
plt.suptitle('Per-Channel Normalization: Centering Each Channel Around Zero', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

After normalization, each channel is centered around 0 with roughly unit variance. This is the same logic as StandardScaler on tabular features: the optimizer works better when inputs are on similar scales and centered.

In practice, many people skip computing dataset-specific statistics and instead use **ImageNet statistics**: `mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]`. These come from the ImageNet dataset (1.2 million images) and work reasonably well for most natural images. When you use a pretrained model (transfer learning), you *must* use ImageNet stats because that's what the model was trained on.

For our bird dataset, we'll use our own computed values since we're training from scratch.

## Part 5: Data Augmentation

This is a concept that has no equivalent in tabular data. You can't meaningfully "augment" a Titanic passenger: what would a horizontally flipped person look like? Their age stays the same, their fare stays the same.

But images have natural invariances. A parrot flipped horizontally is still a parrot. A penguin with slightly brighter colors is still a penguin. We can exploit this to artificially expand our training set.

With only ~200 images per class, augmentation isn't optional here. It's essential. Without it, the model will memorize our small training set almost immediately.

In [ ]:
# Showcase individual augmentations on the same image
sample_path = list((DATA_DIR / 'owl').glob('*'))[5]
sample_pil = Image.open(sample_path).convert('RGB')
sample_pil = sample_pil.resize((200, 200))

augmentations = {
    'Original': transforms.Compose([]),
    'Horizontal Flip': transforms.RandomHorizontalFlip(p=1.0),
    'Rotation (15°)': transforms.RandomRotation(15),
    'Color Jitter': transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    'Random Crop': transforms.RandomResizedCrop(200, scale=(0.7, 1.0)),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (name, aug) in zip(axes, augmentations.items()):
    augmented = aug(sample_pil)
    ax.imshow(augmented)
    ax.set_title(name, fontsize=11)
    ax.axis('off')

plt.suptitle('Individual Augmentations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Each augmentation subtly changes the image while keeping it recognizable. **Color Jitter** is especially useful for bird photos since lighting varies wildly between web-scraped images. **RandomResizedCrop** forces the model to recognize birds that aren't perfectly centered.

Notice we don't use **vertical flip**. An upside-down bird is unusual in nature and could confuse the model. Always think about which augmentations make semantic sense for your domain.

Now let's see what the same image looks like when ALL augmentations are applied together, multiple times:

In [ ]:
# Same image, 9 different augmented versions (what the model sees across epochs)
combined_aug = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
])

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

# First image is the original
axes[0].imshow(sample_pil)
axes[0].set_title('Original', fontweight='bold', fontsize=11)
axes[0].axis('off')

# 9 augmented versions
for i in range(1, 10):
    augmented = combined_aug(sample_pil)
    axes[i].imshow(augmented)
    axes[i].set_title(f'Augmented #{i}', fontsize=10)
    axes[i].axis('off')

plt.suptitle('Same Owl, 9 Different Augmented Versions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Each epoch, the model sees slightly different versions of every image. This prevents it from memorizing exact pixel patterns ("this owl always has bright pixels at position 4,312") and forces it to learn more general features.

Think of it as showing a student the same bird from different angles and in different lighting. They learn what makes an owl an owl, not what makes *this specific photo* an owl.

In [ ]:
# Final transform pipelines: one for training (with augmentation), one for validation (clean)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=BIRD_MEAN, std=BIRD_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(160),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=BIRD_MEAN, std=BIRD_STD),
])

print('Train transform: augment → tensor → normalize')
print('Val transform:   resize → crop → tensor → normalize (no augmentation!)')
print()
print('Why different? We augment training data to prevent overfitting,')
print('but evaluate on clean, unmodified images so our metrics are honest.')

In [ ]:
# Visual verification: check that our transformed images look reasonable
# (This is the equivalent of show_batch() in fastai - always verify before training!)

temp_train = BirdDataset(valid_paths[:8], CLASS_NAMES, transform=train_transform)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i in range(8):
    img, label = temp_train[i]
    # Denormalize for display
    display_img = denormalize(img, BIRD_MEAN, BIRD_STD)
    axes[i].imshow(display_img.permute(1, 2, 0).numpy())
    axes[i].set_title(f'{CLASS_NAMES[label]}', fontsize=11)
    axes[i].axis('off')

plt.suptitle('Pipeline Verification: Augmented + Normalized Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Always verify your pipeline before training. If the images look distorted, overly dark, or unrecognizable, something in your transform chain is wrong. This is the image equivalent of printing `df.head()` after preprocessing tabular data.

The colors might look slightly different from the originals because we denormalized for display (reversing the mean/std normalization). That's fine, the normalized values are what the model actually sees.

## Part 6: Train/Val Split and DataLoaders

In Lesson 06, we split our DataFrame into train and validation sets before any preprocessing. Same principle here, but we need to split our image paths and then create separate Datasets with different transforms (augmentation for training, clean transforms for validation).

In [ ]:
# Shuffle and split paths (80% train, 20% validation)
np.random.shuffle(valid_paths)
split_idx = int(len(valid_paths) * 0.8)
train_paths = valid_paths[:split_idx]
val_paths = valid_paths[split_idx:]

# Create datasets with appropriate transforms
train_dataset = BirdDataset(train_paths, CLASS_NAMES, transform=train_transform)
val_dataset = BirdDataset(val_paths, CLASS_NAMES, transform=val_transform)

print(f'Training:   {len(train_dataset)} images')
print(f'Validation: {len(val_dataset)} images')

# Check class distribution in each split
train_labels = [CLASS_NAMES.index(path[1]) for path in train_paths]
val_labels = [CLASS_NAMES.index(path[1]) for path in val_paths]

print(f'\nTrain class counts: {dict(Counter(path[1] for path in train_paths))}')
print(f'Val class counts:   {dict(Counter(path[1] for path in val_paths))}')

In [ ]:
# Create DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = 4  # load/transform images in parallel background processes

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

# Peek at one batch
images, labels = next(iter(train_loader))
print(f'Batch images shape: {images.shape}')   # (32, 3, 128, 128)
print(f'Batch labels shape: {labels.shape}')    # (32,)
print(f'Labels: {[CLASS_NAMES[l] for l in labels[:6]]}')

Each batch is a tensor of shape `(32, 3, 128, 128)`: 32 images, 3 color channels, 128x128 pixels. The DataLoader handles shuffling and batching for us. `shuffle=True` for training means each epoch presents images in a different order, preventing the model from learning the sequence rather than the content.

## Part 7: Multi-class Classification

Until now, every model you've built answered a yes/no question. "Did this passenger survive?" "Does this person earn over $50k?" One output neuron, sigmoid squashes it to a probability between 0 and 1. That's **binary classification**.

With 5 bird species, we need a fundamentally different output structure. The question isn't "yes or no" — it's "which one?" That's **multi-class classification**, and it changes the tail end of our model.

| | Titanic (binary) | Birds (multi-class) |
|---|---|---|
| Question | Survived? Yes/No | Which bird? (1 of 5) |
| Output neurons | 1 | 5 (one per class) |
| Output range | 0 to 1 (one probability) | 5 probabilities that sum to 1 |
| Activation | Sigmoid | Softmax |
| Loss function | BCELoss | CrossEntropyLoss |

Everything else — the hidden layers, ReLU activations, optimizer, training loop — stays exactly the same. The only change is at the very end of the network.

### Why Not 5 Sigmoids?

You might think: "just have 5 output neurons, each with sigmoid, predicting 'is this an eagle?', 'is this a flamingo?', etc."

Problem: sigmoids are independent. You could get:
- Eagle: 0.8, Flamingo: 0.7, Owl: 0.6

That doesn't make sense. The model is 80% confident it's an eagle AND 70% confident it's a flamingo? These probabilities should compete. If confidence in eagle goes up, confidence in everything else should go down.

That's exactly what **softmax** does: it converts raw numbers into a probability distribution where all values are positive and sum to 1.

In [ ]:
# Let's actually see the problem with 5 independent sigmoids
logits = torch.tensor([2.1, 0.3, 3.8, 1.2, -0.5])

plot_sigmoid_vs_softmax(logits, CLASS_NAMES, CLASS_COLORS)

In [ ]:
# Trace softmax step by step with real numbers
# Suppose the network outputs these raw values (logits) for one image:
logits = np.array([2.1, 0.3, 3.8, 1.2, -0.5])

print('Step 1: Raw logits (direct network output, can be any value)')
for name, val in zip(CLASS_NAMES, logits):
    print(f'  {name:10s}: {val:6.2f}')

print(f'\nStep 2: Exponentiate each value (e^z) — makes everything positive')
exp_logits = np.exp(logits)
for name, raw, exp in zip(CLASS_NAMES, logits, exp_logits):
    print(f'  e^{raw:5.1f} = {exp:8.4f}')

print(f'\nStep 3: Sum all exponentials: {exp_logits.sum():.4f}')

print(f'\nStep 4: Divide each by the sum → probabilities')
softmax_probs = exp_logits / exp_logits.sum()
for name, prob in zip(CLASS_NAMES, softmax_probs):
    bar = '█' * int(prob * 40)
    print(f'  {name:10s}: {prob:.4f}  {bar}')
print(f'\n  Sum: {softmax_probs.sum():.4f} ✓')

plot_softmax_bars(softmax_probs, CLASS_NAMES, CLASS_COLORS)

Three things to notice:

1. **All probabilities are positive** — because e^x is always positive
2. **They sum to 1.0** — because we divide by the total
3. **The largest logit (owl: 3.8) dominates** — softmax amplifies differences. Owl's logit is only ~80% larger than eagle's, but its probability is about 4x higher

The classes compete over a fixed budget of probability (1.0). If owl's confidence goes up, everyone else goes down.

### CrossEntropyLoss

If the true label is "owl" (class 2), CrossEntropyLoss only cares about the probability assigned to owl. High confidence in the correct class = low loss. Low confidence = high loss.

`Loss = -log(probability of correct class)`

In [ ]:
# Trace CrossEntropyLoss with numbers
# Continuing from our softmax output: owl (correct class) got probability 0.8005

correct_class = 2  # owl
p_correct = softmax_probs[correct_class]

print(f'True label: {CLASS_NAMES[correct_class]}')
print(f'Probability assigned to {CLASS_NAMES[correct_class]}: {p_correct:.4f}')
print(f'Loss = -log({p_correct:.4f}) = {-np.log(p_correct):.4f}')

print(f'\nWhat if the model was less confident?')
for p in [0.95, 0.80, 0.50, 0.20, 0.05]:
    loss = -np.log(p)
    bar = '█' * int(loss * 10)
    print(f'  p={p:.2f} → loss={loss:.4f}  {bar}')

print(f'\nPattern: high confidence → low loss, low confidence → high loss')
print(f'And -log(x) goes to infinity as x approaches 0, so the model')
print(f'gets severely punished for being confidently wrong.')

### The Full Pipeline at a Glance

We've traced softmax and cross-entropy separately. Let's see the complete flow for several predictions at once — from raw logits all the way to loss. Think of this like a spreadsheet: each row is one image going through the output layer.

In [ ]:
# Single prediction spreadsheet — trace one owl image through the full pipeline
softmax_spreadsheet([0.5, -1.2, 4.1, 0.3, -0.8], CLASS_NAMES, true_class_idx=2)

One image is nice, but the real intuition comes from comparing predictions. Watch how loss changes depending on whether the model is confident, uncertain, or flat-out wrong:

In [ ]:
# Four scenarios: how does confidence affect the loss?
scenarios = [
    ('Confident & correct',   [0.5, -1.2, 4.1, 0.3, -0.8], 2),  # owl image, high logit for owl
    ('Uncertain but correct',  [1.8, 1.5, 2.1, 1.9, 0.7], 2),   # owl image, logits close together
    ('Confident & WRONG',     [4.0, -0.5, 0.2, 0.1, -1.0], 2),  # owl image, but eagle dominates
    ('Moderate confidence',   [-0.3, -1.0, 0.5, 3.2, -0.5], 3),  # parrot image, decent logit
]

scenario_comparison_table(scenarios, CLASS_NAMES)

### PyTorch Gotcha: CrossEntropyLoss Expects Raw Logits

This is the single most common mistake in PyTorch classification. `nn.CrossEntropyLoss` does softmax + negative log-likelihood **internally**. You pass it the raw logits, NOT softmax probabilities.

```python
# WRONG — double softmax!
probs = torch.softmax(logits, dim=1)
loss = criterion(probs, labels)

# RIGHT — raw logits
loss = criterion(logits, labels)
```

This means your model's last layer should be a plain `nn.Linear(hidden, num_classes)` with no activation. CrossEntropyLoss handles the rest.

In [ ]:
# Verify: our manual softmax + log matches PyTorch's CrossEntropyLoss
logits_tensor = torch.tensor([2.1, 0.3, 3.8, 1.2, -0.5]).unsqueeze(0)  # (1, 5) — batch of 1
target = torch.tensor([2])  # owl

criterion = nn.CrossEntropyLoss()
pytorch_loss = criterion(logits_tensor, target)

# Our manual computation
manual_probs = torch.softmax(logits_tensor, dim=1)
manual_loss = -torch.log(manual_probs[0, target[0]])

print(f'PyTorch CrossEntropyLoss: {pytorch_loss.item():.4f}')
print(f'Manual -log(softmax):     {manual_loss.item():.4f}')
print(f'Match: {abs(pytorch_loss.item() - manual_loss.item()) < 1e-6} ✓')

### What Stays the Same

If multi-class feels like a big shift, here's the reassuring part — almost everything you learned in the tabular lessons carries over unchanged:

- **Same training loop**: forward pass → compute loss → backward pass → update weights
- **Same optimizer**: Adam, with the same learning rate tuning intuition
- **Same hidden layers**: Dense layers with ReLU activations work identically
- **Same concept**: the model outputs numbers, the loss function tells it how wrong it was, gradients flow backward

The *only* structural changes are at the output end: the final layer outputs 5 values instead of 1, and we swap BCELoss for CrossEntropyLoss. That's it. The rest of the model doesn't know or care how many classes there are.

## Part 8: Build the MLP

Time to build the model. Architecturally, this is the same MLP from Lesson 05. The only difference is scale.

In Lesson 05, our Titanic MLP took ~4 input features. Here we need to flatten a 128x128x3 image into a single vector of **49,152** features. That's the same linear algebra — `output = weights @ input + bias` — just with enormous weight matrices.

To feed an image into an MLP, we `nn.Flatten()` it first. This collapses all spatial dimensions into one long vector. The model then treats each pixel value as an independent feature, which means it has zero understanding of spatial relationships. A pixel at position (0, 0) is no different from a pixel at position (64, 64) to the MLP. We'll see why this matters in Part 11.

In [ ]:
INPUT_SIZE = 3 * 128 * 128  # 49,152
HIDDEN_1 = 512
HIDDEN_2 = 256
NUM_CLASSES = 5

class BirdMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(INPUT_SIZE, HIDDEN_1),
            nn.BatchNorm1d(HIDDEN_1),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(HIDDEN_1, HIDDEN_2),
            nn.BatchNorm1d(HIDDEN_2),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(HIDDEN_2, NUM_CLASSES),  # No activation — CrossEntropyLoss handles softmax
        )
        
        # Weight initialization
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.flatten(x)     # (batch, 3, 128, 128) → (batch, 49152)
        return self.layers(x)


model = BirdMLP().to(device)
print(model)

### Weight Initialization

In Lesson 05, we didn't worry about how weights start — PyTorch picked defaults and the Titanic MLP trained fine. With 49,152 input features, initialization matters more.

If all weights start too large, activations explode through the layers. Too small, and gradients vanish — the network barely learns. **Kaiming initialization** (also called He initialization) scales the initial weights based on the number of inputs to each layer, keeping activations in a healthy range as they flow through ReLU layers.

PyTorch's default for `nn.Linear` is actually a simpler uniform initialization. We explicitly use Kaiming because it's designed for ReLU networks. BatchNorm helps compensate for bad initialization (it re-normalizes activations), but good initialization still means faster, more stable early training.

In practice: always use Kaiming for ReLU networks, Xavier/Glorot for sigmoid/tanh networks.

In [ ]:
# Count parameters and compare to our Titanic MLP
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable:        {trainable_params:,}')
print()

# Break down by layer
for name, param in model.named_parameters():
    if 'weight' in name and 'bn' not in name:
        print(f'{name:30s}  shape: {str(list(param.shape)):20s}  params: {param.numel():>12,}')

print(f'\nThe first linear layer alone: {INPUT_SIZE} × {HIDDEN_1} = {INPUT_SIZE * HIDDEN_1:,} weights')
print(f'Compare to Titanic MLP (~4 inputs × 32 hidden = 128 weights)')
print(f"That's a {INPUT_SIZE * HIDDEN_1 // 128:,}x increase. Welcome to image classification.")

In [ ]:
# Verify forward pass with a dummy batch
dummy_batch = torch.randn(4, 3, 128, 128).to(device)
output = model(dummy_batch)
print(f'Input shape:  {dummy_batch.shape}')
print(f'Output shape: {output.shape}')
print(f'Output (logits for first image): {output[0].detach().cpu().numpy().round(4)}')
print(f'\nSoftmax probabilities: {torch.softmax(output[0], dim=0).detach().cpu().numpy().round(4)}')
print(f'Sum: {torch.softmax(output[0], dim=0).sum().item():.4f} ✓')

## Part 9: Training

The training loop follows the same pattern from Lesson 06, with a few additions for image models:

- **`model.train()`** and **`model.eval()`** — these switch BatchNorm and Dropout behavior. During training, Dropout randomly zeros neurons and BatchNorm uses batch statistics. During evaluation, Dropout is disabled and BatchNorm uses running averages. Forgetting `.eval()` is a common source of inconsistent validation results.

- **`torch.no_grad()`** — during evaluation, we don't need gradients. This context manager disables gradient computation, which saves memory and speeds things up.

- **LR finder** — before training, we'll sweep across learning rates to find a good starting point instead of guessing.

In [ ]:
# Simple LR finder: train for a few steps at increasing learning rates
# Watch where the loss drops fastest — that's the sweet spot

def find_lr(model, train_loader, criterion, start_lr=1e-6, end_lr=1, num_steps=80):
    """Sweep learning rates and track loss. Returns lrs and losses."""
    # Save model state so we can restore it after
    initial_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    optimizer = optim.Adam(model.parameters(), lr=start_lr)
    lr_mult = (end_lr / start_lr) ** (1 / num_steps)
    
    lrs, losses = [], []
    current_lr = start_lr
    running_loss = None
    
    model.train()
    data_iter = iter(train_loader)
    
    for step in range(num_steps):
        # Get next batch (cycle if needed)
        try:
            images, labels = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            images, labels = next(data_iter)
        
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        output = model(images)
        loss = criterion(output, labels)
        
        # Smoothed loss for cleaner plot
        if running_loss is None:
            running_loss = loss.item()
        else:
            running_loss = 0.9 * running_loss + 0.1 * loss.item()
        
        lrs.append(current_lr)
        losses.append(running_loss)
        
        # Stop if loss explodes
        if running_loss > losses[0] * 10:
            break
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Increase learning rate
        current_lr *= lr_mult
        for param_group in optimizer.param_groups:
            param_group['lr'] = current_lr
    
    # Restore original model state
    model.load_state_dict(initial_state)
    return lrs, losses


# Reset model for LR finder
model = BirdMLP().to(device)
criterion = nn.CrossEntropyLoss()
lrs, losses = find_lr(model, train_loader, criterion)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lrs, losses, color='#3498db', linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Loss (smoothed)')
ax.set_title('LR Finder: Pick the rate where loss drops fastest')
ax.grid(True, alpha=0.3)

# Mark suggested LR region
best_idx = np.argmin(losses)
suggested_lr = lrs[best_idx] / 5  # Rule of thumb: use 1/5 to 1/10 of the minimum
ax.axvline(suggested_lr, color='#e74c3c', linestyle='--', label=f'Suggested: {suggested_lr:.1e}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Suggested learning rate: {suggested_lr:.1e}')

The LR finder sweeps across learning rates from tiny (1e-6) to large (1). We watch the loss: too small a learning rate barely moves the loss, too large makes it explode. The best spot is where loss decreases most steeply. We pick a rate slightly before the minimum — conservative enough to be stable, aggressive enough to learn fast.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Train for one epoch, return average loss and accuracy."""
    model.train()  # Enable dropout + batch norm training mode
    total_loss, correct, total = 0, 0, 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        predicted = output.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    """Evaluate model, return average loss and accuracy."""
    model.eval()  # Disable dropout + use running batch norm stats
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():  # No gradients needed for evaluation
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            output = model(images)
            loss = criterion(output, labels)
            
            total_loss += loss.item() * images.size(0)
            predicted = output.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / total, correct / total

print('Training functions defined.')
print('Key differences from Lesson 06:')
print('  - model.train() / model.eval() toggle BatchNorm + Dropout')
print('  - torch.no_grad() saves memory during evaluation')
print('  - output.argmax(dim=1) picks the class with highest logit')

### Early Stopping

When training on small datasets, the model eventually starts memorizing training data while validation loss climbs — classic overfitting. We could manually watch the curves and stop at the right moment, but that's tedious and imprecise.

**Early stopping** automates this: track validation loss each epoch, save the model whenever it improves, and stop training if it hasn't improved for `patience` epochs. After stopping, we restore the best saved model — not the final one (which was overfitting).

Two things happen together:
- `ReduceLROnPlateau` (patience=5): shrinks learning rate when validation stalls, giving the optimizer a chance to find a better path
- **Early stopping** (patience=10): pulls the plug if even the reduced learning rate can't help

This lets us set `EPOCHS=50` without worrying about overtraining — early stopping will cut us off at the right time.

In [ ]:
# # Train on CPU first — feel the pain of 49,152 input features
# cpu_model = BirdMLP().to('cpu')
# cpu_criterion = nn.CrossEntropyLoss()
# cpu_optimizer = optim.Adam(cpu_model.parameters(), lr=1e-3)

# import time

# cpu_model.train()
# batch = next(iter(train_loader))
# images, labels = batch[0].to('cpu'), batch[1].to('cpu')

# start = time.time()
# for i in range(3):
#     cpu_optimizer.zero_grad()
#     output = cpu_model(images)
#     loss = cpu_criterion(output, labels)
#     loss.backward()
#     cpu_optimizer.step()
# elapsed = time.time() - start

# print(f'3 batches on CPU: {elapsed:.1f}s ({elapsed/3:.1f}s per batch)')
# print(f'50 epochs × ~{len(train_loader)} batches = ~{50 * len(train_loader)} steps')
# print(f'Estimated full training time on CPU: ~{50 * len(train_loader) * elapsed/3 / 60:.0f} minutes')
# print()
# print('That parameter explosion from Part 8 is not just theoretical — you can feel it.')

# del cpu_model, cpu_criterion, cpu_optimizer  # free memory

In [ ]:
import time


# Now train for real — on GPU
model = BirdMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

print(f'Training on: {device}')

EPOCHS = 50

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

best_val_loss = float('inf')
best_model_state = None
patience = 10
patience_counter = 0

start = time.time()

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = ' ← best'
    else:
        patience_counter += 1
        marker = ''
    
    if (epoch + 1) % 5 == 0 or epoch == 0 or marker:
        print(f'Epoch {epoch+1:2d}/{EPOCHS}  |  '
              f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.3f}  |  '
              f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.3f}{marker}')
    
    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch+1} — no improvement for {patience} epochs')
        break

elapsed = time.time() - start
print(f'\nTraining complete in {elapsed:.1f}s ({elapsed/(epoch+1):.1f}s per epoch)')

model.load_state_dict(best_model_state)
print(f'Restored best model (val loss: {best_val_loss:.4f})')
print(f'Best validation accuracy: {max(history["val_acc"]):.3f} '
      f'(epoch {np.argmax(history["val_acc"])+1})')

In [ ]:
# Training curves
plot_training_curves(history)

The gap between train and validation curves tells the overfitting story. If train accuracy climbs while validation flattens, the model is memorizing training images rather than learning generalizable features. With only ~750 training images and 25M+ parameters, some overfitting is inevitable. This is one reason CNNs work better: they use far fewer parameters by exploiting spatial structure.

`ReduceLROnPlateau` halves the learning rate when validation loss stops improving for 5 epochs. Smaller steps help the optimizer settle into a better minimum rather than bouncing around.

## Part 10: Evaluation

You've used confusion matrices and per-class metrics since L06 and L07 — same tools here, just 5x5 instead of 2x2. What's genuinely new is **top losses analysis**: because we're working with images, we can actually *look at* what the model got wrong and ask "was this a reasonable mistake?"

**Exercise:** Before running the next cell, predict which pair of classes the model confuses most. Think about which birds look similar as flat pixel patterns (remember, the MLP has no spatial awareness — it's working with color distributions, not shapes).

In [ ]:
# Collect all predictions on validation set
all_preds, all_labels, all_probs, all_losses_list = [], [], [], []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        output = model(images)
        probs = torch.softmax(output, dim=1)
        
        # Per-sample loss
        for i in range(len(labels)):
            sample_loss = nn.functional.cross_entropy(output[i:i+1], labels[i:i+1])
            all_losses_list.append(sample_loss.item())
        
        all_preds.extend(output.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_losses_arr = np.array(all_losses_list)

overall_acc = (all_preds == all_labels).mean()
print(f'Overall validation accuracy: {overall_acc:.3f} ({(all_preds == all_labels).sum()}/{len(all_labels)})')

In [ ]:
# Confusion matrix — which birds get confused with which?
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
            cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy
per_class_acc = []
for i, name in enumerate(CLASS_NAMES):
    mask = all_labels == i
    if mask.sum() > 0:
        acc = (all_preds[mask] == all_labels[mask]).mean()
        per_class_acc.append(acc)
        print(f'{name:10s}: {acc:.3f} ({(all_preds[mask] == all_labels[mask]).sum()}/{mask.sum()})')
    else:
        per_class_acc.append(0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(CLASS_NAMES, per_class_acc, color=CLASS_COLORS)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy')
ax.set_ylim(0, 1.1)
ax.axhline(y=overall_acc, color='gray', linestyle='--', alpha=0.7, label=f'Overall: {overall_acc:.1%}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top losses: the images the model was MOST wrong about
# These are the most informative for understanding model failures

# Get validation images for display
val_images_display = []
for path, class_name in val_paths:
    val_images_display.append(path)

# Sort by loss (highest first)
worst_indices = np.argsort(all_losses_arr)[::-1][:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, idx in enumerate(worst_indices):
    img_path = val_images_display[idx]
    img = Image.open(img_path).convert('RGB')
    img = img.resize((128, 128))
    
    actual = CLASS_NAMES[all_labels[idx]]
    predicted = CLASS_NAMES[all_preds[idx]]
    loss_val = all_losses_arr[idx]
    confidence = all_probs[idx].max()
    
    axes[i].imshow(img)
    color = '#2ecc71' if actual == predicted else '#e74c3c'
    axes[i].set_title(f'Actual: {actual}\nPred: {predicted} ({confidence:.0%})\nLoss: {loss_val:.2f}',
                      fontsize=9, color=color, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Top 12 Losses (model\'s worst predictions)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Top losses analysis is one of the most useful debugging tools in image classification (borrowed from fastai's playbook). It tells you:

- **Mislabeled data**: sometimes the image really is wrong in the dataset
- **Ambiguous images**: photos where even a human might struggle (unusual angles, partial views)
- **Systematic failures**: patterns in what the model gets wrong (e.g. always confusing two specific classes)

Look at the images above. Are any of them genuinely hard to classify? Are some of them bad data that shouldn't be in the dataset at all?

In a real project, you'd want to remove or relabel these. The cell below gives you an interactive tool to do exactly that — review the worst images and flag them for removal.

In [ ]:
# Interactive dataset cleaner — inspired by fastai's ImageClassifierCleaner
import ipywidgets as widgets
import shutil

TRASH_DIR = DATA_DIR / '_trash'
TRASH_DIR.mkdir(exist_ok=True)

# Gather all top-loss data into a list of dicts for easy filtering
n_review = 40
worst_indices = np.argsort(all_losses_arr)[::-1][:n_review]

review_items = []
for idx in worst_indices:
    review_items.append({
        'idx': idx,
        'path': val_images_display[idx],
        'actual': CLASS_NAMES[all_labels[idx]],
        'predicted': CLASS_NAMES[all_preds[idx]],
        'loss': all_losses_arr[idx],
        'confidence': all_probs[idx].max(),
        'is_wrong': all_labels[idx] != all_preds[idx],
    })

# Actions: keep, delete, or move to a different class
ACTION_OPTIONS = [('Keep', 'keep'), ('Delete', 'delete')] + \
                 [(f'Move → {c}', f'move_{c}') for c in CLASS_NAMES]

# State
card_widgets = {}  # idx -> action_dropdown

def build_card(item):
    """Build a single image card with thumbnail, info, and action dropdown."""
    idx = item['idx']

    # Thumbnail
    img_out = widgets.Output(layout=widgets.Layout(
        width='260px', height='260px', overflow='hidden'))
    with img_out:
        img = Image.open(item['path']).convert('RGB').resize((260, 260))
        display(img)

    # Status badge
    if item['is_wrong']:
        badge_color, badge_bg = '#fff', '#e74c3c'
        badge_text = f"✗ {item['predicted']} ({item['confidence']:.0%})"
    else:
        badge_color, badge_bg = '#fff', '#f39c12'
        badge_text = f"✓ but loss={item['loss']:.1f}"

    info = widgets.HTML(
        f"<div style='padding:6px; font-size:13px; color:#e0e0e0; line-height:1.5;'>"
        f"<div style='display:inline-block; background:{badge_bg}; color:{badge_color}; "
        f"padding:2px 8px; border-radius:3px; font-size:11px; font-weight:bold; margin-bottom:4px;'>"
        f"{badge_text}</div><br>"
        f"<b>True:</b> {item['actual']}<br>"
        f"<b>Loss:</b> {item['loss']:.2f}</div>"
    )

    # Action dropdown
    action = widgets.Dropdown(
        options=ACTION_OPTIONS, value='keep',
        layout=widgets.Layout(width='250px', height='30px'))
    action.style = {'description_width': '0px'}

    # Card border color
    border_color = '#e74c3c' if item['is_wrong'] else '#f39c12'
    card = widgets.VBox(
        [img_out, info, action],
        layout=widgets.Layout(
            width='280px', padding='8px', margin='4px',
            border=f'2px solid {border_color}',
            border_radius='8px'))

    card_widgets[idx] = action

    def on_action_change(change):
        if change['new'] == 'keep':
            card.layout.border = f'2px solid {border_color}'
            card.layout.opacity = '1'
        elif change['new'] == 'delete':
            card.layout.border = '2px solid #e74c3c'
            card.layout.opacity = '0.5'
        else:
            card.layout.border = '2px solid #3498db'
            card.layout.opacity = '0.8'
        update_summary()

    action.observe(on_action_change, names='value')
    return card

# Filter dropdown
filter_dd = widgets.Dropdown(
    options=[('All classes', 'all')] + [(c.capitalize(), c) for c in CLASS_NAMES],
    value='all', description='Filter:',
    layout=widgets.Layout(width='200px'))

# Grid container — horizontal scroll, wraps into rows
grid_container = widgets.VBox(
    layout=widgets.Layout(
        max_height='750px', overflow_y='auto',
        padding='5px', width='100%'))

# Summary bar
summary = widgets.HTML(layout=widgets.Layout(margin='8px 0'))

def update_summary():
    n_delete = sum(1 for a in card_widgets.values() if a.value == 'delete')
    n_move = sum(1 for a in card_widgets.values() if a.value.startswith('move_'))
    parts = []
    if n_delete:
        parts.append(f"<span style='color:#e74c3c; font-weight:bold;'>{n_delete} to delete</span>")
    if n_move:
        parts.append(f"<span style='color:#3498db; font-weight:bold;'>{n_move} to relabel</span>")
    if not parts:
        summary.value = "<span style='color:#888;'>No changes flagged yet.</span>"
    else:
        summary.value = ' &nbsp;|&nbsp; '.join(parts)

def render_grid(class_filter='all'):
    items = review_items if class_filter == 'all' else \
            [it for it in review_items if it['actual'] == class_filter]
    cards = [build_card(it) for it in items]
    # Single horizontal row per batch, scrollable horizontally
    row = widgets.HBox(cards, layout=widgets.Layout(
        overflow_x='auto', flex_flow='row nowrap', width='100%'))
    grid_container.children = [row]
    update_summary()

def on_filter(change):
    card_widgets.clear()
    render_grid(change['new'])

filter_dd.observe(on_filter, names='value')

# Apply button
btn_apply = widgets.Button(
    description='Apply Changes', button_style='danger', icon='check',
    layout=widgets.Layout(width='200px', height='36px'))
result_msg = widgets.HTML()

def on_apply(b):
    deleted, moved = 0, 0
    for idx, action_dd in card_widgets.items():
        val = action_dd.value
        if val == 'keep':
            continue
        item = next(it for it in review_items if it['idx'] == idx)
        src_path = Path(item['path'])
        if not src_path.exists():
            continue
        if val == 'delete':
            dest = TRASH_DIR / src_path.name
            if dest.exists():
                dest = TRASH_DIR / f'{src_path.stem}_{deleted}{src_path.suffix}'
            shutil.move(str(src_path), str(dest))
            deleted += 1
        elif val.startswith('move_'):
            target_class = val.replace('move_', '')
            dest_dir = DATA_DIR / target_class
            dest_dir.mkdir(exist_ok=True)
            shutil.move(str(src_path), str(dest_dir / src_path.name))
            moved += 1

    parts = []
    if deleted:
        parts.append(f'{deleted} deleted (moved to _trash)')
    if moved:
        parts.append(f'{moved} relabeled')
    result_msg.value = (
        f"<div style='color:#2ecc71; font-weight:bold; padding:6px;'>"
        f"{'  |  '.join(parts)}<br>"
        f"<span style='font-weight:normal; color:#e0e0e0;'>"
        f"Re-run data loading (Parts 2-6) and retrain to see the effect.</span></div>")

btn_apply.on_click(on_apply)

# Header
header = widgets.HTML(
    "<div style='font-size:15px; font-weight:bold; color:#e0e0e0; padding:4px 0;'>"
    "Dataset Cleaner</div>"
    "<div style='font-size:11px; color:#aaa; padding-bottom:6px;'>"
    "Sorted by loss (worst first). Red border = wrong prediction. "
    "Orange border = correct but confused. Scroll right to see more. "
    "Change the dropdown to flag images.</div>")

# Assemble
toolbar = widgets.HBox([filter_dd], layout=widgets.Layout(padding='4px 0'))
bottom_bar = widgets.HBox([btn_apply, summary, result_msg],
                          layout=widgets.Layout(align_items='center', padding='6px 0'))

render_grid()
display(widgets.VBox([header, toolbar, grid_container, bottom_bar]))

In [ ]:
# Visualize some correct and incorrect predictions side by side
correct_mask = all_preds == all_labels
incorrect_mask = ~correct_mask

correct_indices = np.where(correct_mask)[0]
incorrect_indices = np.where(incorrect_mask)[0]

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

# Top row: correct predictions (high confidence)
correct_conf = all_probs[correct_indices].max(axis=1)
best_correct = correct_indices[np.argsort(correct_conf)[::-1][:5]]

for i, idx in enumerate(best_correct):
    img = Image.open(val_images_display[idx]).convert('RGB').resize((128, 128))
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'{CLASS_NAMES[all_labels[idx]]}\n✓ {all_probs[idx].max():.0%}',
                         fontsize=10, color='#2ecc71', fontweight='bold')
    axes[0, i].axis('off')

# Bottom row: incorrect predictions
np.random.seed(42)
if len(incorrect_indices) >= 5:
    sample_incorrect = np.random.choice(incorrect_indices, 5, replace=False)
else:
    sample_incorrect = incorrect_indices

for i, idx in enumerate(sample_incorrect):
    img = Image.open(val_images_display[idx]).convert('RGB').resize((128, 128))
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'Actual: {CLASS_NAMES[all_labels[idx]]}\nPred: {CLASS_NAMES[all_preds[idx]]} ({all_probs[idx].max():.0%})',
                         fontsize=9, color='#e74c3c', fontweight='bold')
    axes[1, i].axis('off')

# Hide unused axes if fewer than 5 incorrect
for i in range(len(sample_incorrect), 5):
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Correct ✓', fontsize=13, fontweight='bold', color='#2ecc71')
axes[1, 0].set_ylabel('Incorrect ✗', fontsize=13, fontweight='bold', color='#e74c3c')
plt.suptitle('Model Predictions: Best Correct vs Incorrect', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Correct: {correct_mask.sum()}/{len(correct_mask)} | Incorrect: {incorrect_mask.sum()}/{len(incorrect_mask)}')

## Part 11: Why MLPs Struggle with Images

Our MLP treats every pixel as an independent feature. It has no concept of "next to", "above", or "part of a shape." To the MLP, a pixel at position (0,0) in the top-left corner could just as easily be at position (127,127) in the bottom-right. All spatial relationships are lost the moment we `nn.Flatten()`.

To prove this, let's run an experiment: **shuffle all the pixels in every image** and retrain. If the MLP truly ignores spatial structure, shuffling shouldn't hurt much.

**Exercise:** Predict before running. Will the shuffled MLP get roughly the same accuracy, or dramatically worse?

In [ ]:
# What a shuffled image looks like
sample_img, sample_label = val_dataset[0]
display_img = denormalize(sample_img, BIRD_MEAN, BIRD_STD)

# Create a fixed permutation (same shuffle for all images)
torch.manual_seed(42)
perm = torch.randperm(128 * 128)

# Shuffle pixels: reshape to (3, N), permute columns, reshape back
shuffled = display_img.view(3, -1)[:, perm].view(3, 128, 128)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(display_img.permute(1, 2, 0).numpy())
axes[0].set_title(f'Original: {CLASS_NAMES[sample_label]}', fontsize=12)
axes[1].imshow(shuffled.permute(1, 2, 0).numpy())
axes[1].set_title('Same image, pixels shuffled', fontsize=12)
for ax in axes:
    ax.axis('off')
plt.suptitle('Pixel Shuffle: Destroying Spatial Structure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('To a human, the shuffled image is pure noise.')
print('To an MLP, it\'s just the same 49,152 numbers in a different order.')

In [ ]:
# Train an MLP on shuffled images
# We need custom datasets that apply the same pixel shuffle

class ShuffledBirdDataset(Dataset):
    def __init__(self, image_paths, class_names, transform, perm):
        self.image_paths = image_paths
        self.class_to_idx = {name: i for i, name in enumerate(class_names)}
        self.transform = transform
        self.perm = perm
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path, class_name = self.image_paths[idx]
        image = Image.open(path).convert('RGB')
        label = self.class_to_idx[class_name]
        image = self.transform(image)
        # Shuffle pixels (same permutation for all images)
        image = image.view(3, -1)[:, self.perm].view(3, 128, 128)
        return image, label

shuffled_train = ShuffledBirdDataset(train_paths, CLASS_NAMES, train_transform, perm)
shuffled_val = ShuffledBirdDataset(val_paths, CLASS_NAMES, val_transform, perm)
shuffled_train_loader = DataLoader(shuffled_train, batch_size=BATCH_SIZE, shuffle=True)
shuffled_val_loader = DataLoader(shuffled_val, batch_size=BATCH_SIZE, shuffle=False)

# Train shuffled model (same architecture, same epochs)
shuffled_model = BirdMLP().to(device)
shuffled_criterion = nn.CrossEntropyLoss()
shuffled_optimizer = optim.Adam(shuffled_model.parameters(), lr=1e-3)
shuffled_scheduler = optim.lr_scheduler.ReduceLROnPlateau(shuffled_optimizer, mode='min', factor=0.5, patience=5)

shuffled_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('Training on SHUFFLED images...')
for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(shuffled_model, shuffled_train_loader, shuffled_criterion, shuffled_optimizer)
    v_loss, v_acc = evaluate(shuffled_model, shuffled_val_loader, shuffled_criterion)
    shuffled_scheduler.step(v_loss)
    
    shuffled_history['train_loss'].append(t_loss)
    shuffled_history['val_loss'].append(v_loss)
    shuffled_history['train_acc'].append(t_acc)
    shuffled_history['val_acc'].append(v_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS}  |  '
              f'Train Acc: {t_acc:.3f}  |  Val Acc: {v_acc:.3f}')

print(f'\nBest shuffled val accuracy: {max(shuffled_history["val_acc"]):.3f}')

In [ ]:
# Compare: original vs shuffled
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['val_acc'], label='Original', color='#3498db', linewidth=2)
axes[0].plot(shuffled_history['val_acc'], label='Shuffled Pixels', color='#e74c3c', linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('Original vs Shuffled Pixels')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Bar chart comparison
best_original = max(history['val_acc'])
best_shuffled = max(shuffled_history['val_acc'])
bars = axes[1].bar(['Original', 'Shuffled'], [best_original, best_shuffled], 
                    color=['#3498db', '#e74c3c'], width=0.5)
for bar, val in zip(bars, [best_original, best_shuffled]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.1%}', ha='center', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Best Validation Accuracy')
axes[1].set_title('Best Accuracy Comparison')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

diff = abs(best_original - best_shuffled)
print(f'Original:  {best_original:.1%}')
print(f'Shuffled:  {best_shuffled:.1%}')
print(f'Difference: {diff:.1%}')

The result is striking: the MLP performs similarly on shuffled images because it was never using spatial information in the first place. It classifies based on color distributions and pixel statistics, not shapes or edges.

This is the fundamental limitation of MLPs for vision. A flamingo's pink color palette is recognizable whether the pixels are arranged as a bird shape or randomly scattered. But distinguishing an eagle from an owl based purely on color? Much harder.

### The Parameter Explosion Problem

The other issue is efficiency. An MLP must connect every input pixel to every neuron in the first hidden layer. As images get bigger, this scales quadratically:

In [ ]:
# Parameter explosion: first layer params at different resolutions
resolutions = [32, 64, 128, 224, 512]
hidden = 512

sizes = []
for res in resolutions:
    input_features = 3 * res * res
    first_layer_params = input_features * hidden
    sizes.append(first_layer_params)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([f'{r}x{r}' for r in resolutions], [s / 1e6 for s in sizes], 
              color=['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6'])
for bar, s in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{s/1e6:.1f}M', ha='center', fontweight='bold')
ax.set_xlabel('Image Resolution')
ax.set_ylabel('First Layer Parameters (millions)')
ax.set_title('MLP Parameter Explosion: First Layer Only')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'At 224x224 (standard ImageNet size): {3*224*224*512:,} params in just the first layer')
print(f'A typical CNN like ResNet-18 has 11.7M params TOTAL across 18 layers')
print(f'Our MLP needs {3*224*224*512/1e6:.0f}M for a single layer. Wildly inefficient.')

### The CNN Teaser

The solution to both problems — no spatial awareness AND parameter explosion — is the **Convolutional Neural Network (CNN)**. Instead of connecting every pixel to every neuron, CNNs use small filters (e.g. 3x3) that slide across the image. This means:

1. **Spatial awareness**: A 3x3 filter sees neighboring pixels, so it can detect edges, corners, and shapes
2. **Parameter efficiency**: The same 3x3 filter (just 27 parameters!) is reused across the entire image
3. **Translation invariance**: A cat in the top-left looks the same to a CNN as a cat in the bottom-right

A CNN can achieve 90%+ accuracy on bird classification with fewer total parameters than our MLP's first layer alone. That's what we'll build in the CNN lesson.

For now, the key insight is: **MLPs can classify images, but they do it the hard way — by memorizing color statistics, ignoring everything about shape and structure.**

## Summary

We built a complete image classification pipeline from scratch. Here's every step we covered:

**Data Pipeline**
1. Loaded images from folder structure (class = folder name)
2. Explored class distribution (imbalanced dataset)
3. Surveyed image sizes (all different, need standardizing)
4. Cleaned data (verified every image loads, removed corrupted files)
5. Built resize strategy (Resize + CenterCrop vs RandomResizedCrop)
6. Created `transforms.Compose()` pipeline
7. Computed per-channel normalization (mean/std per RGB channel)
8. Added data augmentation (flip, rotation, color jitter, random crop)
9. Verified pipeline visually (always check before training!)
10. Split into train/validation with different transforms

**Model & Training**
11. Built custom `Dataset` class for on-the-fly loading
12. Created `DataLoader` for batching and shuffling
13. Multi-class classification: softmax + CrossEntropyLoss (not sigmoid!)
14. Built MLP architecture with BatchNorm and Dropout
15. Used LR finder to pick learning rate
16. Trained with `model.train()`/`model.eval()` and `torch.no_grad()`

**Evaluation**
17. Confusion matrix (which classes confuse each other)
18. Per-class accuracy (reveals imbalanced performance)
19. Top losses analysis (most informative failures)
20. Correct vs incorrect prediction visualization

**MLP Limitations**
21. Pixel shuffle experiment (MLP ignores spatial structure)
22. Parameter explosion (impractical for large images)

**Next up**: Practice these concepts on CIFAR-10, then build a CNN that actually understands what it's looking at.

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>